<a href="https://colab.research.google.com/github/adamapadane/gestor-tareas-iep/blob/main/Caso_practico_Unidad_2_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning — Caso práctico Unidad 2
**IEP | CNN y CGAN**

Ejecutar de arriba abajo en Google Colab con GPU. Los resultados y las conclusiones empíricas se generan al ejecutar; no se han simulado métricas.

## 1.- Implementación de una CNN para clasificación de imágenes

En esta primera sección, el alumno debe de implementar una red neuronal convolucional para clasificación de imágenes con el dataset CIFAR-100, que consiste en imágenes de 100 clases de objetos diferentes, frene a los 10 que usaba su dataset hermano, CIFAR-10.

Antes de empezar con el ejercicio, es importante plantearnos las siguientes preguntas, que son parte de la evaluación de la tarea y que habrá que responder en el documento de la tarea de los temas 3-4:

- 1. ¿Necesitaré una arquitectura igual a la utilizada para CIFAR-10?
- 2. ¿Será un problema más o menos complicado para la red neuronal?
- 3. A la hora de evaluar las métricas, ¿debemos interpretarlas de la misma manera que cuando usamos CIFAR-10?
- 4. Si ya tengo una arquitectura válida para el dataset CIFAR-10, ¿qué es lo mínimo que tengo que cambiar de la red para poder entrenar la nueva red con CIFAR-100?

Para complicar más el reto, se limita el número de capas ocultas (incluyendo capas convolucionales y densas) a 6 (la capa de salida no cuenta).

*Nota: el dataset CIFAR-100 es un dataset muy complejo de trabajar, ya que tiene de una gran cantidad de posibles clases (100). No te preocupes si no consigues un accuracy alto; a más clases, más complicado es el problema. Con un prototipo rápido a penas he alcanzado un 36% de accuracy.*




### Respuestas teóricas
1. No necesariamente. Ambas bases tienen imágenes RGB de 32 × 32, por lo que se puede reutilizar el extractor de características. Sin embargo, CIFAR-100 puede beneficiarse de más filtros y regularización.
2. Es más complicado: hay más categorías, menos ejemplos por clase (500 frente a 5.000 en entrenamiento) y similitudes visuales entre clases.
3. La accuracy conserva su definición, pero su contexto cambia: el azar uniforme pasa del 10 % al 1 %. No se deben comparar porcentajes sin considerar la dificultad, la distribución y los errores por clase.
4. Cambiar la salida de 10 a 100 neuronas softmax y adaptar la codificación de etiquetas a 100 categorías. La entropía cruzada categórica sigue siendo válida; es necesario volver a entrenar el modelo.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'
import tensorflow as tf
import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt
keras.utils.set_random_seed(42)
print('TensorFlow:', tf.__version__, 'Keras:', keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Separación estratificada: 450 ejemplos de entrenamiento y 50 de validación por clase.
(x, y), (x_test, y_test) = keras.datasets.cifar100.load_data(label_mode='fine')
n_classes = 100
rng = np.random.default_rng(42)
train_idx, val_idx = [], []
for k in range(n_classes):
    idx = rng.permutation(np.flatnonzero(y.ravel() == k))
    val_idx.extend(idx[:50]); train_idx.extend(idx[50:])
x_train, x_val = x[train_idx], x[val_idx]
y_train = keras.utils.to_categorical(y[train_idx], n_classes)
y_val = keras.utils.to_categorical(y[val_idx], n_classes)
y_test = keras.utils.to_categorical(y_test, n_classes)
# Normalización y aumentación solo en entrenamiento, fuera del modelo.
def make_ds(images, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if training: ds = ds.shuffle(10000, seed=42)
    ds = ds.batch(64)
    def prep(a, b):
        a = tf.cast(a, tf.float32) / 255.
        if training:
            a = tf.image.random_flip_left_right(a)
            a = tf.pad(a, [[0,0],[4,4],[4,4],[0,0]], mode='REFLECT')
            a = tf.image.random_crop(a, [tf.shape(a)[0],32,32,3])
        return a, b
    return ds.map(prep, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
train_ds = make_ds(x_train, y_train, True)
val_ds = make_ds(x_val, y_val)
test_ds = make_ds(x_test, y_test)

### Arquitectura
Se adopta una interpretación conservadora: seis capas ocultas totales, incluyendo el pooling. Las convoluciones con stride 2 reducen la resolución sin añadir capas de pooling intermedias. La regularización L2 está integrada en las capas.

In [ ]:
reg = keras.regularizers.l2(1e-4)
model = keras.Sequential([
    keras.Input(shape=(32,32,3)),
    layers.Conv2D(64, 3, padding='same', activation='relu', kernel_regularizer=reg),
    layers.Conv2D(128, 3, strides=2, padding='same', activation='relu', kernel_regularizer=reg),
    layers.Conv2D(192, 3, padding='same', activation='relu', kernel_regularizer=reg),
    layers.Conv2D(256, 3, strides=2, padding='same', activation='relu', kernel_regularizer=reg),
    layers.Conv2D(384, 3, padding='same', activation='relu', kernel_regularizer=reg),
    layers.GlobalAveragePooling2D(),
    layers.Dense(n_classes, activation='softmax')
], name='CNN_CIFAR100')
assert len(model.layers)-1 == 6
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top5')])
model.summary()
EPOCHS_CNN = 60
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CNN, callbacks=[
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)])

In [ ]:
results = model.evaluate(test_ds, return_dict=True)
print(results)
fig, ax = plt.subplots(1,2,figsize=(12,4))
for a, metric in zip(ax,['accuracy','loss']):
    a.plot(history.history[metric], label='Entrenamiento')
    a.plot(history.history['val_'+metric], label='Validación')
    a.set(xlabel='Época', title=metric); a.legend()
plt.show()
model.save('cnn_cifar100.keras')

In [ ]:
from IPython.display import display, Markdown
h = history.history
best = int(np.argmin(h['val_loss']))
gap = h['accuracy'][best]-h['val_accuracy'][best]
display(Markdown(f"""### Conclusiones del ejercicio 1
La red convolucional ha obtenido una exactitud de test del **{results['accuracy']:.2%}** y una exactitud top-5 del {results['top5']:.2%}. La pérdida de test ha sido {results['loss']:.3f}, incluyendo la regularización. La mejor época según la pérdida de validación fue la {best+1}, con una exactitud de validación del {h['val_accuracy'][best]:.2%}. La diferencia entre entrenamiento y validación en dicha época fue de {gap*100:.2f} puntos porcentuales. Este indicador debe interpretarse junto con las curvas, pues la aumentación modifica la dificultad de los ejemplos de entrenamiento. La arquitectura respeta el máximo de seis capas ocultas y utiliza convoluciones para extraer características espaciales. La separación de validación permite ajustar el entrenamiento sin consultar el conjunto de test. La exactitud debe contextualizarse considerando las cien categorías y el nivel de azar del 1 %. Como trabajo futuro, sería interesante comparar distintas anchuras de filtros y estrategias de regularización usando exclusivamente validación, manteniendo la restricción de profundidad."""))

## 2.- Implementación de una CGAN para generación de imágenes

En este apartado, crearemos una CGAN, similar a la vista en el notebook *conditional-gan.ipynb*, pero para generar imágenes sintéticas a partir del dataset Fashion MNIST. Aquí, nuevamente, cabe preguntarnos varios temas antes de empezar, que también deberán responderse en el fichero de entrega de la tarea 3-4:

- 5. Desde el punto de vista técnico, sin fijarnos aún en la calidad de las imágenes sintetizadas, ¿habría que hacer algún cambio obligatorio con respecto a la red utilizada para sintetizar datos de MNIST? Justifique su respuesta.
- 6. A priori, ¿piensas que la red generadora y la discriminadora deberán más complejas, igual de complejas, o menos complejas que en el caso de la red utilizada con MNIST para sintetizar datos de calidad suficiente? Justifique su respuesta.

*Nota: aquí queremos generar imágenes de muy baja resolución de elementos mucho más complejos que simples dígitos. No os preocupéis si los resultados son lo suficientemente buenos, lo importante es conseguir la*


### Respuestas teóricas
5. No es obligatorio modificar la arquitectura ni las dimensiones: FashionMNIST y MNIST tienen diez clases e imágenes de 28 × 28 × 1. Sí hay que sustituir la carga del dataset y actualizar el significado de las etiquetas. Si se conserva la salida sigmoide, los píxeles deben permanecer en [0,1].
6. Es razonable prever que se necesite mayor capacidad, porque las prendas presentan texturas, siluetas y variaciones más complejas. No es una obligación matemática: la calidad depende también del equilibrio adversarial, la optimización y el número de épocas. Se propone aumentar la capacidad de forma moderada y comprobar visualmente la diversidad y coherencia de las clases.

In [ ]:
batch_size = 64
num_channels, num_classes, image_size, latent_dim = 1, 10, 28, 128
(x_fashion, y_fashion), (x_fashion_test, y_fashion_test) = keras.datasets.fashion_mnist.load_data()
# Se reserva el test; no es necesario para entrenar la CGAN.
all_clothes = x_fashion.astype('float32')[...,None]/255.
all_labels = keras.utils.to_categorical(y_fashion, num_classes).astype('float32')
dataset = tf.data.Dataset.from_tensor_slices((all_clothes, all_labels)).shuffle(10000, seed=42).batch(batch_size).prefetch(tf.data.AUTOTUNE)
generator_in_channels = latent_dim + num_classes
discriminator_in_channels = num_channels + num_classes
print(generator_in_channels, discriminator_in_channels)
class_names = ['Camiseta','Pantalón','Jersey','Vestido','Abrigo','Sandalia','Camisa','Zapatilla','Bolso','Botín']

In [ ]:
# Adaptación de conditional_gan.ipynb: salida binaria sin activación (logits).
discriminator = keras.Sequential([
    keras.Input((28,28,discriminator_in_channels)),
    layers.Conv2D(64,3,strides=2,padding='same'), layers.LeakyReLU(0.2),
    layers.Conv2D(128,3,strides=2,padding='same'), layers.LeakyReLU(0.2),
    layers.Conv2D(256,3,padding='same'), layers.LeakyReLU(0.2),
    layers.Flatten(), layers.Dropout(0.3), layers.Dense(1)
], name='discriminator')
generator = keras.Sequential([
    keras.Input((generator_in_channels,)),
    layers.Dense(7*7*128), layers.BatchNormalization(), layers.LeakyReLU(0.2),
    layers.Reshape((7,7,128)),
    layers.Conv2DTranspose(128,4,strides=2,padding='same'), layers.BatchNormalization(), layers.LeakyReLU(0.2),
    layers.Conv2DTranspose(64,4,strides=2,padding='same'), layers.BatchNormalization(), layers.LeakyReLU(0.2),
    layers.Conv2D(1,7,padding='same',activation='sigmoid')
], name='generator')
generator.summary(); discriminator.summary()
assert generator.output_shape[1:] == (28,28,1)

In [ ]:
class ConditionalGAN(keras.Model):
    def __init__(self, discriminator, generator, latent_dim):
        super().__init__()
        self.discriminator, self.generator = discriminator, generator
        self.latent_dim = latent_dim
        self.gen_loss_tracker = keras.metrics.Mean(name='generator_loss')
        self.disc_loss_tracker = keras.metrics.Mean(name='discriminator_loss')
    @property
    def metrics(self):
        return [self.gen_loss_tracker, self.disc_loss_tracker]
    def compile(self, d_optimizer, g_optimizer, loss_fn):
        super().compile()
        self.d_optimizer, self.g_optimizer, self.loss_fn = d_optimizer, g_optimizer, loss_fn
    def train_step(self, data):
        real_images, labels = data
        bs = tf.shape(real_images)[0]
        # Cada etiqueta se replica espacialmente, conservando el orden de canales.
        maps = tf.tile(labels[:,None,None,:], [1,28,28,1])
        def sample():
            return tf.concat([tf.random.normal([bs,self.latent_dim]), labels], axis=1)
        # Convención del notebook original: 1 = sintética, 0 = real.
        fake = tf.stop_gradient(self.generator(sample(), training=False))
        combined = tf.concat([tf.concat([fake,maps],-1), tf.concat([real_images,maps],-1)],0)
        targets = tf.concat([tf.ones([bs,1]),tf.zeros([bs,1])],0)
        with tf.GradientTape() as tape:
            d_loss = self.loss_fn(targets,self.discriminator(combined, training=True))
        weights = self.discriminator.trainable_weights
        self.d_optimizer.apply_gradients(zip(tape.gradient(d_loss,weights),weights))
        # Solo se actualiza el generador; intenta que sus imágenes parezcan reales.
        with tf.GradientTape() as tape:
            fake = self.generator(sample(), training=True)
            pred = self.discriminator(tf.concat([fake,maps],-1), training=False)
            g_loss = self.loss_fn(tf.zeros([bs,1]),pred)
        weights = self.generator.trainable_weights
        self.g_optimizer.apply_gradients(zip(tape.gradient(g_loss,weights),weights))
        self.gen_loss_tracker.update_state(g_loss)
        self.disc_loss_tracker.update_state(d_loss)
        return {metric.name:metric.result() for metric in self.metrics}

In [ ]:
# Ruido fijo: permite comparar la evolución sin cambiar las muestras.
fixed_noise = tf.random.stateless_normal([30,latent_dim],seed=[42,1])
fixed_labels = tf.one_hot(np.tile(np.arange(10),3),10)
fixed_input = tf.concat([fixed_noise,fixed_labels],1)
def show_samples(gen, title):
    images = gen(fixed_input,training=False).numpy()
    fig, axes = plt.subplots(3,10,figsize=(15,5))
    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i,:,:,0],cmap='gray',vmin=0,vmax=1)
        if i<10: ax.set_title(class_names[i],fontsize=8)
        ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.show()
class Monitor(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch+1)%10==0: show_samples(self.model.generator,f'Época {epoch+1}')
cond_gan = ConditionalGAN(discriminator,generator,latent_dim)
cond_gan.compile(d_optimizer=keras.optimizers.Adam(0.0002,beta_1=0.5),
                 g_optimizer=keras.optimizers.Adam(0.0002,beta_1=0.5),
                 loss_fn=keras.losses.BinaryCrossentropy(from_logits=True))
EPOCHS_GAN = 50
history_gan = cond_gan.fit(dataset,epochs=EPOCHS_GAN,callbacks=[Monitor()])

In [ ]:
plt.figure(figsize=(9,4))
for metric, values in history_gan.history.items(): plt.plot(values,label=metric)
plt.xlabel('Época'); plt.ylabel('Entropía cruzada'); plt.legend(); plt.show()
show_samples(generator,'Muestras finales: tres ejemplos por clase')
generator.save('generator_fashion.keras')
discriminator.save('discriminator_fashion.keras')

### Conclusiones del ejercicio 2 — completar tras la ejecución
La adaptación de la GAN condicional permite solicitar imágenes de una categoría concreta mediante la concatenación de ruido y etiquetas en el generador, y de mapas de etiquetas e imágenes en el discriminador. Se ha conservado la resolución de FashionMNIST y la normalización del ejemplo original. Las capas transpuestas reconstruyen progresivamente la imagen y la normalización por lotes busca estabilizar el generador. El entrenamiento alterna las actualizaciones de ambas redes sin modificar los pesos del discriminador durante el paso del generador. Las curvas de pérdida no bastan para determinar la calidad visual: deben analizarse las cuadrículas de ruido fijo, comprobando nitidez, diversidad dentro de cada clase y correspondencia con las etiquetas.

**Observación pendiente:** tras ejecutar, indicar qué prendas se reconocen mejor, cuáles se confunden, si aparecen muestras repetidas y cómo evoluciona la calidad entre las épocas 10 y 50. No se afirma ningún resultado visual sin observar las imágenes.

### Referencias y cambios
Material IEP: enunciado Unidad 2, plantilla y conditional_gan.ipynb facilitados. Se ha reorganizado la plantilla en celdas, sustituido la instalación de dependencias innecesarias, completado el paso adversarial y corregido la expansión espacial de etiquetas. No se han realizado experimentos comparativos ni entrenamiento en este entorno.